# 15 · Q5 signal or artifact: profile the SQLite database and test the +250% claim

Queries `inventory_and_sales.db` with `sqlite3` + pandas. First check: does the database itself create an artifact (mixed date formats), and what does "+250%" depend on?

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = next(d for d in ('data', '../data', '../../data') if os.path.isdir(d))
MISSING_TOKENS = {'', 'null', 'n/a', 'na', 'nan', 'none'}
DECISION_LOG = []

def log_decision(step, decision, rows_touched, rows_before=None, rows_after=None):
    DECISION_LOG.append({'step': step, 'decision': decision, 'rows_touched': int(rows_touched),
                         'rows_before': rows_before, 'rows_after': rows_after})


In [2]:
CATEGORY_MAP = {
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'electronics': 'Electronics',
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'health': 'Health', 'health & wellness': 'Health',
    'household': 'Household', 'household essentials': 'Household',
    'snacks': 'Snacks',
}
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
              '%d-%b-%Y %H:%M', '%m-%d-%Y %H:%M:%S']

def load_transactions():
    return pd.read_csv(os.path.join(DATA_DIR, 'walmart_transactions.csv'), dtype=str)

def standardize_missing(df):
    """Strip whitespace; turn '', NULL, N/A, NaN... into real NaN."""
    out = df.copy()
    touched = 0
    for c in out.columns:
        s = out[c].str.strip()
        newly_na = s.str.lower().isin(MISSING_TOKENS) & out[c].notna()
        touched += int(newly_na.sum())
        out[c] = s.mask(newly_na)
    log_decision('missing', 'strip whitespace; blank/NULL/N-A tokens -> NaN (cells, not rows)', touched, len(df), len(out))
    return out

def remove_duplicates(df):
    n0 = len(df)
    out = df.drop_duplicates()
    log_decision('duplicates', 'drop exact duplicate rows (keep first)', n0 - len(out), n0, len(out))
    n1 = len(out)
    out = out.drop_duplicates('transaction_id')
    log_decision('duplicates', 'drop repeated transaction_id (keep first)', n1 - len(out), n1, len(out))
    return out

def clean_unit_price(df):
    out = df.copy()
    raw = out['unit_price']
    dollar = int(raw.str.contains('$', regex=False, na=False).sum())
    out['unit_price'] = pd.to_numeric(raw.str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
    bad = int(out['unit_price'].isna().sum() - raw.isna().sum())
    log_decision('unit_price', "remove '$', cast text -> float", dollar, len(df), len(out))
    log_decision('unit_price', 'values that still failed numeric cast -> NaN', bad, len(df), len(out))
    return out

def clean_quantity(df):
    out = df.copy()
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    n0 = len(out)
    out = out[out['quantity'] > 0]
    log_decision('quantity', 'drop rows with negative/zero/unparseable quantity (entry errors; sign-flip not assumed)', n0 - len(out), n0, len(out))
    return out

def parse_timestamps(df):
    out = df.copy()
    ts = pd.Series(pd.NaT, index=out.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        todo = ts.isna() & out['timestamp'].notna()
        ts[todo] = pd.to_datetime(out.loc[todo, 'timestamp'], format=fmt, errors='coerce')
    unparsed = int((ts.isna() & out['timestamp'].notna()).sum())
    missing = int(out['timestamp'].isna().sum())
    out['timestamp'] = ts
    log_decision('timestamp', f'parse {len(TS_FORMATS)} mixed formats into one datetime (naive local time); {unparsed} unparseable', int(ts.notna().sum()), len(df), len(out))
    log_decision('timestamp', 'missing timestamps kept (NaT) but excluded from any date-based analysis; not imputed', missing, len(df), len(out))
    return out

def clean_store_id(df):
    out = df.copy()
    raw = out['store_id']
    out['store_id'] = (raw.str.upper().str.replace('_', '-', regex=False)
                          .str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True))
    log_decision('store_id', "uppercase, strip, '_'/missing '-' -> canonical 'FL-###'", int((raw != out['store_id']).sum()), len(df), len(out))
    return out

def clean_category(df):
    out = df.copy()
    raw = out['category']
    out['category'] = raw.str.lower().map(CATEGORY_MAP)
    log_decision('category', f'collapse {raw.nunique()} spellings into {out["category"].nunique()} canonical categories', int((raw != out['category']).sum()), len(df), len(out))
    return out

def clean_transactions():
    DECISION_LOG.clear()
    df = load_transactions()
    for step in (standardize_missing, remove_duplicates, clean_unit_price, clean_quantity,
                 parse_timestamps, clean_store_id, clean_category):
        df = step(df)
    df['product_name_raw'] = df['product_name']
    return df.reset_index(drop=True)


In [3]:
import sqlite3
con = sqlite3.connect(os.path.join(DATA_DIR, 'inventory_and_sales.db'))
print(pd.read_sql("SELECT name, sql FROM sqlite_master WHERE type='table'", con).to_string())

# Problem 1: mixed date formats in daily_sales_summary
print('\ndate string lengths:'); print(pd.read_sql('SELECT length(date) AS len, COUNT(*) AS n FROM daily_sales_summary GROUP BY len', con))
q_week = "SELECT COUNT(*) AS n FROM daily_sales_summary WHERE category = 'Entertainment' AND {col} BETWEEN {a} AND {b}"
naive = pd.read_sql(q_week.format(col='date', a="'2024-09-19'", b="'2024-09-25'"), con)['n'][0]
fixed = pd.read_sql(q_week.format(col="REPLACE(date, '-', '')", a="'20240919'", b="'20240925'"), con)['n'][0]
print(f'Entertainment rows in Sept 19-25 (expect 12 stores x 7 days = 84): naive date filter = {naive}, after normalising = {fixed}')
assert fixed == 84

# Entertainment daily units straight from SQL, dates normalised
ds = pd.read_sql("SELECT REPLACE(date, '-', '') AS d, SUM(total_units) AS units FROM daily_sales_summary "
                 "WHERE category = 'Entertainment' GROUP BY d ORDER BY d", con)
ent_db = ds.assign(day=pd.to_datetime(ds['d'], format='%Y%m%d')).set_index('day')['units']
assert len(ent_db) == 30

def surge_stats(s):
    base = s.loc['2024-09-01':'2024-09-15'].mean()
    return {'baseline_avg': round(base, 1),
            'week_avg_pct': round((s.loc['2024-09-19':'2024-09-25'].mean() / base - 1) * 100, 1),
            'peak_day_pct': round((s.max() / base - 1) * 100, 1), 'peak_day': str(s.idxmax().date())}

good = pd.read_sql("SELECT date, SUM(total_units) AS units FROM daily_sales_summary WHERE category = 'Entertainment' AND length(date) = 10 GROUP BY date", con)
ent_naive = good.assign(day=pd.to_datetime(good['date'])).set_index('day')['units']
print('\nEntertainment surge, SQL totals with dates normalised:', surge_stats(ent_db))
print('Entertainment surge if the 218 YYYYMMDD rows are silently dropped:', surge_stats(ent_naive))


                  name                                                                                                                                                                        sql
0  daily_sales_summary                               CREATE TABLE daily_sales_summary (\n    date TEXT,\n    store_id TEXT,\n    category TEXT,\n    total_units REAL,\n    total_revenue REAL\n)
1     inventory_levels  CREATE TABLE inventory_levels (\n    date TEXT,\n    store_id TEXT,\n    sku TEXT,\n    units_on_hand INTEGER,\n    units_on_order INTEGER,\n    reorder_point INTEGER\n)

date string lengths:
   len     n
0    8   218
1   10  3382
Entertainment rows in Sept 19-25 (expect 12 stores x 7 days = 84): naive date filter = 78, after normalising = 84

Entertainment surge, SQL totals with dates normalised: {'baseline_avg': np.float64(568.5), 'week_avg_pct': np.float64(145.3), 'peak_day_pct': np.float64(271.7), 'peak_day': '2024-09-24'}
Entertainment surge if the 218 YYYYMMDD rows are silent

**What this shows.** The database has its own date artifact: 218 of 3,600 rows use `YYYYMMDD` instead of `YYYY-MM-DD`, so a plain SQL date filter silently loses 6 of the 84 Entertainment rows in the surge week. Fixing it matters little here (week average +145% vs +148% if the rows are dropped). The board-games number also depends on the definition: Entertainment is +145% on a 7-day average but +272% on its single best day (Sept 24), so a "+250%" figure is only reachable as a peak-day number, not a weekly one. Note this is the whole Entertainment category (Monopoly plus playing cards).